# Induction heads - finding your first transformer circuit

Step 3 of 6 in the mech interp curriculum.

We train a tiny 2-layer attention-only transformer on a task that only induction can solve: predict the next token in a sequence formed by repeating a random first half. After training, we'll inspect the attention patterns of the 8 heads and find the canonical 2-head circuit - a previous-token head in Layer 0 plus an induction head in Layer 1 - that implements pattern-matching.

Read `README.md` first. It explains the circuit. This notebook is the runnable companion.

Expected runtime on Colab T4:.

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

torch.manual_seed(0)
np.random.seed(0)

## 2. Hyperparameters

In [ ]:
# task
d_vocab    = 64           # alphabet size — tokens are integers in [0, 63]
half_len   = 25
seq_len    = 2 * half_len # full sequence: random half + repeated half

# model
d_model    = 64
n_heads    = 4
d_head     = 16           # d_model = n_heads * d_head
n_layers   = 2            # new in this step!

# training
batch_size  = 128
n_steps     = 3_000
lr          = 1e-3
weight_decay = 0.01
log_every   = 100

## 3. Data generation - random repeated sequences

Each example is a fresh random first half (length 25) concatenated with itself. This task is unsolvable for the first half (tokens are uniformly random) and trivially solved by induction in the second half ("I've seen this token before - predict what came after it last time").

In [ ]:
def make_batch(batch_size, half_len, d_vocab, device):
    first  = torch.randint(0, d_vocab, (batch_size, half_len), device=device)
    second = first.clone()
    return torch.cat([first, second], dim=1)   # (batch_size, 2*half_len)

# sanity check
sample = make_batch(1, half_len, d_vocab, device)[0].cpu().tolist()
print(f'first half:  {sample[:half_len]}')
print(f'second half: {sample[half_len:]}')
print(f'identical?   {sample[:half_len] == sample[half_len:]}')

## 4. The 2-layer attention-only transformer

Same `Attention` class as in step 2, now wrapped in a loop over 2 layers. No MLPs (we want the circuit to be visible without clutter). No layer norm (cleaner for mech interp).

We also need a causal mask in the attention: token at position `i` should only attend to positions `≤ i`. (Step 2 didn't bother - sequence length was 3 and the task didn't care. Here it does.)

In [ ]:
class Attention(nn.Module):
    def __init__(self, d_model, n_heads, d_head):
        super().__init__()
        self.W_Q = nn.Parameter(torch.randn(n_heads, d_model, d_head) * (1.0 / d_model**0.5))
        self.W_K = nn.Parameter(torch.randn(n_heads, d_model, d_head) * (1.0 / d_model**0.5))
        self.W_V = nn.Parameter(torch.randn(n_heads, d_model, d_head) * (1.0 / d_model**0.5))
        self.W_O = nn.Parameter(torch.randn(n_heads, d_head, d_model) * (1.0 / d_model**0.5))
        self.d_head = d_head

    def forward(self, x, return_pattern=False):
        # x: (batch, pos, d_model)
        q = torch.einsum('bpd,hde->bphe', x, self.W_Q)
        k = torch.einsum('bpd,hde->bphe', x, self.W_K)
        v = torch.einsum('bpd,hde->bphe', x, self.W_V)

        scores = torch.einsum('bphe,bqhe->bhpq', q, k) / (self.d_head ** 0.5)

        # causal mask: position i can only attend to positions ≤ i
        T = x.shape[1]
        causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        scores = scores.masked_fill(causal_mask, float('-inf'))

        attn = F.softmax(scores, dim=-1)   # (batch, head, query_pos, key_pos)
        z    = torch.einsum('bhpq,bqhe->bphe', attn, v)
        out  = torch.einsum('bphe,hed->bpd',   z, self.W_O)
        if return_pattern:
            return out, attn
        return out

class Transformer(nn.Module):
    def __init__(self, d_vocab, d_model, n_heads, d_head, n_layers, seq_len):
        super().__init__()
        self.W_E   = nn.Parameter(torch.randn(d_vocab, d_model) * (1.0 / d_model**0.5))
        self.W_pos = nn.Parameter(torch.randn(seq_len, d_model) * (1.0 / d_model**0.5))
        self.attns = nn.ModuleList([Attention(d_model, n_heads, d_head) for _ in range(n_layers)])
        self.W_U   = nn.Parameter(torch.randn(d_model, d_vocab) * (1.0 / d_model**0.5))

    def forward(self, tokens):
        x = self.W_E[tokens] + self.W_pos
        for attn in self.attns:
            x = x + attn(x)
        return x @ self.W_U

    def forward_with_patterns(self, tokens):
        """Same as forward but also returns each layer's attention pattern."""
        x = self.W_E[tokens] + self.W_pos
        patterns = []
        for attn in self.attns:
            out, pat = attn(x, return_pattern=True)
            x = x + out
            patterns.append(pat)
        return x @ self.W_U, patterns

In [ ]:
model = Transformer(d_vocab, d_model, n_heads, d_head, n_layers, seq_len).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params:,}')

## 5. Training

Standard AdamW. Loss is cross-entropy on next-token prediction across all positions, but we log loss on the first half (unsolvable) and the second half (induction-solvable) separately so we can see the gap.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

first_losses, second_losses, log_steps = [], [], []

for step in range(n_steps + 1):
    tokens = make_batch(batch_size, half_len, d_vocab, device)   # (B, 2*half_len)
    logits = model(tokens)                                       # (B, 2*half_len, d_vocab)

    # predict token[t+1] from position t — shift by 1
    pred   = logits[:, :-1, :]    # (B, 2*half_len - 1, d_vocab)
    target = tokens[:, 1:]        # (B, 2*half_len - 1)

    loss = F.cross_entropy(pred.reshape(-1, d_vocab), target.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % log_every == 0:
        with torch.no_grad():
            # split loss into first-half and second-half positions
            per_pos = F.cross_entropy(
                pred.reshape(-1, d_vocab),
                target.reshape(-1),
                reduction='none',
            ).reshape(batch_size, -1)                     # (B, 2*half_len - 1)
            first  = per_pos[:, :half_len - 1].mean().item()
            second = per_pos[:,  half_len - 1:].mean().item()
        first_losses.append(first)
        second_losses.append(second)
        log_steps.append(step)
        if step % 500 == 0:
            print(f'step {step:5d}  | first-half loss {first:.3f}  | second-half loss {second:.3f}')

print('\nTraining complete.')

## 6. Loss curves - the wow moment

First-half loss stays near `log(64) ≈ 4.16` (the model literally cannot predict a uniform-random next token). Second-half loss drops toward zero. The gap is the model learning induction.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(log_steps, first_losses,  label='first half (unsolvable — uniform random)', linewidth=2)
plt.plot(log_steps, second_losses, label='second half (solvable by induction)',     linewidth=2)
plt.axhline(np.log(d_vocab), color='gray', linestyle='--', linewidth=1,
            label=f'random-guess level  log(64) = {np.log(d_vocab):.2f}')
plt.xlabel('step')
plt.ylabel('loss')
plt.title('Induction emerges: second-half loss drops, first-half loss stays at random level')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Find the previous-token head and the induction head

We score each head against two reference patterns and plot the scores. Whichever Layer 0 head has the highest previous-token score is the previous-token head; whichever Layer 1 head has the highest induction score is the induction head.

In [ ]:
# grab attention patterns on a fresh batch
with torch.no_grad():
    eval_tokens = make_batch(256, half_len, d_vocab, device)
    _, patterns = model.forward_with_patterns(eval_tokens)   # list of (B, H, P, P)
    patterns = [p.mean(dim=0).cpu().numpy() for p in patterns]  # average over batch → (H, P, P) each

def previous_token_score(pattern):
    """How much does each position attend to the position immediately before it?
    pattern shape: (P, P).  We average over positions 1..P-1.
    """
    P = pattern.shape[0]
    return np.mean([pattern[i, i - 1] for i in range(1, P)])

def induction_score(pattern, half_len):
    """On the second half (positions half_len..P-1), how much does position
    half_len+k attend to position k+1 (the token after the previous occurrence)?
    """
    P = pattern.shape[0]
    scores = []
    for k in range(half_len - 1):
        # destination position in the second half
        dst = half_len + k
        # source position: the one right after the matching first-half occurrence
        src = k + 1
        if dst < P:
            scores.append(pattern[dst, src])
    return float(np.mean(scores))

# compute scores for every head
all_heads = []
for layer_idx, layer_pat in enumerate(patterns):
    for h in range(n_heads):
        pat = layer_pat[h]
        all_heads.append({
            'layer': layer_idx,
            'head':  h,
            'prev_score':       previous_token_score(pat),
            'induction_score':  induction_score(pat, half_len),
            'pattern':          pat,
        })

# print a table
print(f'{"layer":>5}  {"head":>4}  {"prev-token score":>18}  {"induction score":>17}')
for h in all_heads:
    print(f'{h["layer"]:>5}  {h["head"]:>4}  {h["prev_score"]:>18.3f}  {h["induction_score"]:>17.3f}')

In [ ]:
# bar chart of both scores
labels = [f'L{h["layer"]}H{h["head"]}' for h in all_heads]
prev   = [h['prev_score']      for h in all_heads]
ind    = [h['induction_score'] for h in all_heads]

x = np.arange(len(all_heads))
width = 0.4

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width / 2, prev, width, label='previous-token score')
ax.bar(x + width / 2, ind,  width, label='induction score')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('score (0 = no signal, 1 = perfect)')
ax.set_title('Which heads do what?')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# identify the winners
prev_winner = max([h for h in all_heads if h['layer'] == 0], key=lambda h: h['prev_score'])
ind_winner  = max([h for h in all_heads if h['layer'] == 1], key=lambda h: h['induction_score'])
print(f'\nLayer-0 previous-token head: L{prev_winner["layer"]}H{prev_winner["head"]}  (score {prev_winner["prev_score"]:.3f})')
print(f'Layer-1 induction head:      L{ind_winner["layer"]}H{ind_winner["head"]}  (score {ind_winner["induction_score"]:.3f})')

## 8. Visualise the attention patterns of all 8 heads

An attention pattern is a 50×50 matrix where row `i` (the query position) shows where position `i` is attending to (with attention summing to 1 across the row; positions to the right of the diagonal are masked since attention is causal).

Focus first on the two heads identified above. You should see:

- **Previous-token head**: a bright stripe one cell below the diagonal - each position attends to the one before it.
- **Induction head**: in the bottom-right 25×25 quadrant, a clear diagonal stripe shifted by ~25 positions - each second-half position attends to the position immediately after the matching first-half token.


In [ ]:
fig, axes = plt.subplots(n_layers, n_heads, figsize=(3 * n_heads, 3 * n_layers))
for layer_idx in range(n_layers):
    for h in range(n_heads):
        ax = axes[layer_idx, h]
        pat = patterns[layer_idx][h]
        ax.imshow(pat, cmap='viridis', aspect='auto', vmin=0, vmax=pat.max())
        title = f'L{layer_idx}H{h}'
        if layer_idx == prev_winner['layer'] and h == prev_winner['head']:
            title += '  ← previous-token'
        if layer_idx == ind_winner['layer']  and h == ind_winner['head']:
            title += '  ← INDUCTION'
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('key position')
        ax.set_ylabel('query position')
        # mark the halfway line
        ax.axhline(half_len - 0.5, color='red', linewidth=0.5, alpha=0.6)
        ax.axvline(half_len - 0.5, color='red', linewidth=0.5, alpha=0.6)
plt.suptitle('Attention patterns of all 8 heads (averaged over a batch of repeated sequences)', y=1.02)
plt.tight_layout()
plt.show()

## 9. Discussion

You've just found a real circuit. The previous-token head and induction head together implement an algorithm - in-context pattern matching - that neither could do alone. This is your first hands-on encounter with circuits, the main object of study in transformer mech interp.

Things to note:

- The two heads form an inductive pair via K-composition: the induction head's keys are built from the residual stream after Layer 0, which carries information the previous-token head wrote in. Strip out either head (with weight ablation or activation patching - step 4) and the circuit breaks.
- The 6 other heads aren't doing nothing - they often play supporting roles (smoothing, copying, etc.) - but on this synthetic task they don't have a clean specialisation. In real LLMs every head ends up doing something, but the work is messier.
- In production language models, induction heads form during a sharp phase transition in training, around the same time the model becomes capable of in-context learning. Olsson et al. 2022 makes this case at length.

What we did not do (this is the topic of step 4):

- We visualised attention patterns and inferred function from them. That's correlational - "the pattern looks right." We did not causally verify that these heads are responsible. The standard tool is activation patching: replace this head's output with one from a different prompt and see whether the model still gets the right answer. Step 4 (IOI in GPT-2 small) introduces this.

Onwards to step 4.